# PREPARACIÓN DEL CÓDIGO DE PRODUCCIÓN

## IMPORTAR LAS LIBRERIAS

Actualizar a las que se usen finalmente

In [2]:
#Posibles paquetes necesarios para el desarrollo del proyecto. No harán falta todos siempre. Seleccionar los necesarios
import os
import numpy as np
import pandas as pd
import pickle  
import cloudpickle #Mismas funciones de pickle de guardar modelos + funciones de guardado de las funciones específicasimport seaborn as sns
import sqlalchemy as sa
import scipy as sp
import matplotlib.pyplot as plt
%matplotlib inline
%config IPCompleter.greedy = True

#Formato sin notación científica
pd.options.display.float_format = '{:15.2f}'.format 

#Automcompletar rápido
%config IPCompleter.greedy=True

#Desactivar la notación científica
pd.options.display.float_format = '{:.2f}'.format

#Desactivar los warnings
import warnings
warnings.filterwarnings("ignore")

#Mostrar el máximo de filas posibles de una tabla
pd.set_option('display.max_rows', 100) #Número de filas que deben verse. None = Máx

#Mostrar mas caracteres de las columnas. Se usa cuando se corta el texto
pd.set_option('display.max_colwidth', 20) #Número de caractres que deben verse. None = Máx

#Representación visual de un pipeline
from sklearn import set_config
set_config(display = 'diagram') #diagram/text 

#Transformación de variables
from janitor import clean_names
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import TargetEncoder
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.preprocessing import Binarizer
from sklearn.preprocessing import PowerTransformer
from sklearn.preprocessing import QuantileTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import MaxAbsScaler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler

#Modelos ML: CLASIFICACIÓN
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
#Métrica de error: Habrá que incluir la que necesitemos
from sklearn.metrics import roc_auc_score

#Modelos ML: REGRESIÓN
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
#Métrica de error: Habrá que incluir la que necesitemos
from sklearn.metrics import mean_absolute_error

#Modelos ML: NO SUPERVISADO
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics import calinski_harabasz_score
from sklearn.metrics import davies_bouldin_score

## CARGAR LOS DATOS

### PRECIO_M2 IDEALISTA

In [3]:
#Ruta del proyecto
ruta_proyecto = 'C:/Users/Oscar/OneDrive - FM4/Escritorio/Python Data Mastery/EstructuraDirectorio/03_MACHINE_LEARNING/08_CASOS/007_AIRBNB'

#Nombre del fichero de datos
nombre_fichero_datos = 'precios_idealista_Dic24.csv'

#Cargar los datos
ruta_completa = ruta_proyecto + '/02_Datos/01_Originales/' + nombre_fichero_datos
precio_m2 = pd.read_csv(ruta_completa)
precio_m2

,table__cell,table__cell 2,table__cell 3,table__cell 4,table__cell 5,table__cell 6,table__cell 7,icon-elbow,icon-elbow href
0,Madrid,4.514 €/m2,"+ 0,9 %","+ 4,1 %","+ 12,4 %",4.514 €/m2 jun 2024,"0,0 %",NaN,NaN
1,NaN,4.850 €/m2,"+ 2,0 %","+ 5,6 %","+ 10,4 %",4.850 €/m2 jun 2024,"0,0 %",Arganzuela,https://www.idea...
2,NaN,3.686 €/m2,"+ 0,3 %","+ 3,9 %","+ 5,1 %",3.686 €/m2 mar 2009,"0,0 %",Barajas,https://www.idea...
3,"- 14,5 %",2.712 €/m2,"+ 1,4 %","+ 3,8 %","+ 10,4 %",3.173 €/m2 jun 2007,NaN,Carabanchel,https://www.idea...
4,"- 0,6 %",6.170 €/m2,"+ 0,4 %","+ 14,9 %","- 0,8 %",6.221 €/m2 abr 2024,NaN,Centro,https://www.idea...
5,NaN,6.285 €/m2,"+ 1,1 %","+ 4,1 %","+ 10,2 %",6.285 €/m2 jun 2024,"0,0 %",Chamartín,https://www.idea...
6,NaN,6.665 €/m2,"+ 0,5 %","+ 5,0 %","+ 12,2 %",6.665 €/m2 jun 2024,"0,0 %",Chamberí,https://www.idea...
7,NaN,3.676 €/m2,"+ 0,9 %","+ 5,1 %","+ 8,7 %",3.676 €/m2 jun 2024,"0,0 %",Ciudad Lineal,https://www.idea...
8,NaN,4.038 €/m2,"+ 0,8 %","+ 3,3 %","+ 5,4 %",4.038 €/m2 jun 2024,"0,0 %",Fuencarral,https://www.idea...
9,NaN,4.368 €/m2,"+ 0,8 %","+ 5,7 %","+ 9,6 %",4.368 €/m2 jun 2024,"0,0 %",Hortaleza,https://www.idea...


In [4]:
def modificaciones_idealista(precio_m2):
    precio_m2 = precio_m2.loc[1:,['table__cell 2','icon-elbow']] \
    .rename(columns = {'table__cell 2':'precio_m2','icon-elbow':'neighbourhood_group'})
    
    precio_m2['precio_m2'] = precio_m2['precio_m2'].str.replace(' €/m2','').str.replace('.','').astype('int')
    
    return(precio_m2)
precio_m2 = modificaciones_idealista(precio_m2)
precio_m2

,precio_m2,neighbourhood_group
1,4850,Arganzuela
2,3686,Barajas
3,2712,Carabanchel
4,6170,Centro
5,6285,Chamartín
6,6665,Chamberí
7,3676,Ciudad Lineal
8,4038,Fuencarral
9,4368,Hortaleza
10,2772,Latina


In [5]:
#Pipeline con las modificaciones hechas en la tabla
idealista_ft = FunctionTransformer(modificaciones_idealista)

pipe_idealista = make_pipeline(idealista_ft)

nombre_pipe = 'pipe_idealista.pickle'

ruta_pipe = ruta_proyecto + '/04_Modelos/' + nombre_pipe

with open(ruta_pipe, mode='wb') as file:
   cloudpickle.dump(pipe_idealista, file)

### LISTINGS AIRBNB

In [6]:
#Ruta del proyecto
ruta_proyecto = 'C:/Users/Oscar/OneDrive - FM4/Escritorio/Python Data Mastery/EstructuraDirectorio/03_MACHINE_LEARNING/08_CASOS/007_AIRBNB'

con = sa.create_engine('sqlite:///C:/Users/Oscar/OneDrive - FM4/Escritorio/Python Data Mastery/EstructuraDirectorio/03_MACHINE_LEARNING/08_CASOS/007_AIRBNB/02_Datos/01_Originales/airbnb2025.db')

#Nombre del fichero de datos
nombre_fichero_datos = 'airbnb2025.db'

#Cargar los datos
ruta_completa = ruta_proyecto + '/02_Datos/01_Originales/' + nombre_fichero_datos
listings = pd.read_sql('listings', con).drop(columns='index')#.set_index('id')
listings

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365,number_of_reviews_ltm,license
0,21853,Bright and airy ...,83531,Abdel,Latina,Cármenes,40.40,-3.74,Private room,31.00,4,33,2018-07-15,0.27,2,144,0,None
1,30320,Great Vacational...,130907,Dana,Centro,Sol,40.41,-3.70,Entire home/apt,NaN,5,172,2022-09-26,0.98,3,0,0,None
2,30959,Beautiful loft i...,132883,Angela,Centro,Embajadores,40.41,-3.70,Entire home/apt,NaN,3,8,2017-05-30,0.07,1,0,0,None
3,40916,Holiday Apartmen...,130907,Dana,Centro,Universidad,40.42,-3.71,Entire home/apt,NaN,5,49,2021-12-11,0.29,3,0,0,None
4,62423,MAGIC ARTISTIC H...,303845,Arturo,Centro,Justicia,40.42,-3.70,Private room,69.00,1,219,2024-11-24,2.73,3,332,44,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26755,1308816654087147793,Vive Madrid Río ...,251333188,Jose Maria,Latina,Puerta del Angel,40.41,-3.72,Entire home/apt,258.00,1,0,None,NaN,19,350,0,None
26756,1308818613904246041,Alquilo habitaci...,118339834,Brandon,Centro,Palacio,40.41,-3.71,Private room,20.00,15,0,None,NaN,1,364,0,None
26757,1308839477505592914,Sierra Habitacio...,23441165,Andrea,Puente de Vallecas,Numancia,40.40,-3.66,Private room,69.00,1,0,None,NaN,26,309,0,None
26758,1308842708819314249,Estudio luminoso...,251333188,Jose Maria,Latina,Puerta del Angel,40.41,-3.72,Entire home/apt,258.00,1,0,None,NaN,19,348,0,None


### LISTINGS_DET AIRBNB

In [7]:
#Ruta del proyecto
ruta_proyecto = 'C:/Users/Oscar/OneDrive - FM4/Escritorio/Python Data Mastery/EstructuraDirectorio/03_MACHINE_LEARNING/08_CASOS/007_AIRBNB'

con = sa.create_engine('sqlite:///C:/Users/Oscar/OneDrive - FM4/Escritorio/Python Data Mastery/EstructuraDirectorio/03_MACHINE_LEARNING/08_CASOS/007_AIRBNB/02_Datos/01_Originales/airbnb2025.db')

#Nombre del fichero de datos
nombre_fichero_datos = 'airbnb2025.db'

#Cargar los datos
ruta_completa = ruta_proyecto + '/02_Datos/01_Originales/' + nombre_fichero_datos
listings_det = pd.read_sql('listings_det', con).drop(columns='index')#.set_index('id')
listings_det

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,21853,https://www.airb...,20241212051353,2024-12-12,city scrape,Bright and airy ...,We have a quiet ...,We live in a lea...,https://a0.musca...,83531,...,4.82,4.21,4.67,None,f,2,0,2,0,0.27
1,30320,https://www.airb...,20241212051353,2024-12-12,previous scrape,Great Vacational...,None,None,https://a0.musca...,130907,...,4.78,4.90,4.69,None,f,3,3,0,0,0.98
2,30959,https://www.airb...,20241212051353,2024-12-12,previous scrape,Beautiful loft i...,Beautiful Loft 6...,None,https://a0.musca...,132883,...,4.63,4.88,4.25,None,f,1,1,0,0,0.07
3,40916,https://www.airb...,20241212051353,2024-12-12,previous scrape,Holiday Apartmen...,None,None,https://a0.musca...,130907,...,4.79,4.88,4.55,None,f,3,3,0,0,0.29
4,62423,https://www.airb...,20241212051353,2024-12-12,city scrape,MAGIC ARTISTIC H...,INCREDIBLE HOME ...,DISTRICT WITH VE...,https://a0.musca...,303845,...,4.85,4.97,4.58,None,f,3,1,2,0,2.73
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26755,1308816654087147793,https://www.airb...,20241212051353,2024-12-12,city scrape,Vive Madrid Río ...,Escape to the he...,"Madrid, Communit...",https://a0.musca...,251333188,...,NaN,NaN,NaN,None,t,19,19,0,0,NaN
26756,1308818613904246041,https://www.airb...,20241212051353,2024-12-12,city scrape,Alquilo habitaci...,"Hi! I'm Monica, ...",None,https://a0.musca...,118339834,...,NaN,NaN,NaN,None,f,1,0,1,0,NaN
26757,1308839477505592914,https://www.airb...,20241212051353,2024-12-12,city scrape,Sierra Habitacio...,Tour the most po...,None,https://a0.musca...,23441165,...,NaN,NaN,NaN,None,t,26,0,23,0,NaN
26758,1308842708819314249,https://www.airb...,20241212051353,2024-12-12,city scrape,Estudio luminoso...,"Renovated, cozy ...",None,https://a0.musca...,251333188,...,NaN,NaN,NaN,None,t,19,19,0,0,NaN


In [8]:
pd.set_option('display.max_columns', None)
def preparacion_df(datos):
    listings_det, listings, precio_m2 = datos
    
    a_eliminar = [
     'listing_url',
     'scrape_id',
     'last_scraped',
     'source',
     'description',
     'neighborhood_overview',
     'picture_url',
     'host_url',
     'host_name',
     'host_location',
     'host_about',
     'host_picture_url',
     'host_neighbourhood',
     'neighbourhood',
     'neighbourhood_group_cleansed',
     'minimum_minimum_nights',
     'maximum_minimum_nights',
     'minimum_maximum_nights',
     'maximum_maximum_nights',
     'minimum_nights_avg_ntm',
     'maximum_nights_avg_ntm',
     'calendar_updated',
     'calendar_last_scraped',
     'host_thumbnail_url']
    listings_det = listings_det.drop(columns = a_eliminar)    
    
    # 1. Normalización del nombre de barrios
    def join_listings(listings_det, listings):
        listings['neighbourhood_group'] = listings['neighbourhood_group'].replace({
            'Fuencarral - El Pardo': 'Fuencarral',
            'Moncloa - Aravaca': 'Moncloa',
            'San Blas - Canillejas': 'San Blas'
        })
        listings_det = pd.merge(left=listings_det, right=listings[['id', 'neighbourhood_group']], how='left', on='id')
        return listings_det

    listings_det = join_listings(listings_det, listings)

    # 2. Join con precios por metro cuadrado
    def join_df(listings_det, precio_m2):
        df = pd.merge(left=listings_det, right=precio_m2, how='left', on='neighbourhood_group')
        return df
    df = join_df(listings_det, precio_m2)
    return df

datos = (listings_det, listings, precio_m2)
df = preparacion_df(datos)
df

,id,name,host_id,host_since,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms,beds,amenities,price,minimum_nights,maximum_nights,has_availability,availability_30,availability_60,availability_90,availability_365,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month,neighbourhood_group,precio_m2
0,21853,Bright and airy ...,83531,2010-02-21,None,None,0%,f,2.00,2.00,"['email', 'phone']",t,t,Cármenes,40.40,-3.74,Private room in ...,Private room,1,1.00,1 bath,1.00,1.00,"[""First aid kit""...",$31.00,4,40,t,0,0,0,144,33,0,0,2014-10-10,2018-07-15,4.58,4.72,4.56,4.75,4.82,4.21,4.67,None,f,2,0,2,0,0.27,Latina,2772
1,30320,Great Vacational...,130907,2010-05-24,None,None,None,f,3.00,6.00,"['email', 'phone']",t,f,Sol,40.41,-3.70,Entire rental unit,Entire home/apt,2,NaN,1 bath,1.00,NaN,"[""Heating"", ""Wif...",None,5,180,None,0,0,0,0,172,0,0,2010-07-06,2022-09-26,4.63,4.71,4.88,4.82,4.78,4.90,4.69,None,f,3,3,0,0,0.98,Centro,6170
2,30959,Beautiful loft i...,132883,2010-05-26,None,None,None,f,1.00,4.00,"['email', 'phone']",t,f,Embajadores,40.41,-3.70,Entire loft,Entire home/apt,2,NaN,1 bath,1.00,NaN,"[""Breakfast"", ""H...",None,3,730,None,0,0,0,0,8,0,0,2015-05-12,2017-05-30,4.38,4.14,4.38,4.63,4.63,4.88,4.25,None,f,1,1,0,0,0.07,Centro,6170
3,40916,Holiday Apartmen...,130907,2010-05-24,None,None,None,f,3.00,6.00,"['email', 'phone']",t,f,Universidad,40.42,-3.71,Entire rental unit,Entire home/apt,3,NaN,1 bath,1.00,NaN,"[""Heating"", ""Wif...",None,5,180,None,0,0,0,0,49,0,0,2010-11-01,2021-12-11,4.65,4.69,4.90,4.85,4.79,4.88,4.55,None,f,3,3,0,0,0.29,Centro,6170
4,62423,MAGIC ARTISTIC H...,303845,2010-11-29,within an hour,100%,100%,f,3.00,3.00,"['email', 'phone']",t,t,Justicia,40.42,-3.70,Private room in ...,Private room,4,1.50,1.5 shared baths,1.00,2.00,"[""Books and read...",$69.00,1,30,t,13,37,59,332,219,44,3,2018-05-10,2024-11-24,4.64,4.78,4.42,4.79,4.85,4.97,4.58,None,f,3,1,2,0,2.73,Centro,6170
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26755,1308816654087147793,Vive Madrid Río ...,251333188,2019-03-26,within a few hours,100%,67%,None,163.00,234.00,"['email', 'phone']",t,t,Puerta del Angel,40.41,-3.72,Entire rental unit,Entire home/apt,4,1.00,1 bath,1.00,2.00,"[""Wifi"", ""Kitche...",$258.00,1,365,t,15,45,75,350,0,0,0,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,t,19,19,0,0,NaN,Latina,2772
26756,1308818613904246041,Alquilo habitaci...,118339834,2017-02-27,None,None,None,f,1.00,1.00,"['email', 'phone']",t,t,Palacio,40.41,-3.71,Private room in ...,Private room,1,1.00,1 shared bath,1.00,1.00,"[""Wifi"", ""Kitche...",$20.00,15,16,t,29,59,89,364,0,0,0,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,f,1,0,1,0,NaN,Centro,6170
26757,1308839477505592914,Sierra Habitacio...,23441165,2014-11-07,within an hour,95%,98%,f,39.00,90.00,"['email', 'phone']",t,t,Numancia,40.40,-3.66,Room in hotel,Private room,2,1.00,1 private bath,1.00,1.00,"[""Exterior secur...",$69.00,1,365,t,14,44,74,309,0,0,0,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,t,26,0,23,0,NaN,Puente de Vallecas,2367
26758,1308842708819314249,Estudio luminoso...,251333188,2019-03-26,within a few hours,100%,67%,None,163.00,234.00,"['email', 'phone']",t,t,Puerta del Angel,40.41,-3.72,Entire rental

In [9]:
#Pipeline con las modificaciones hechas en la tabla
preparacion_df_ft = FunctionTransformer(preparacion_df)

pipe_preparacion_df = make_pipeline(preparacion_df_ft)

nombre_pipe = 'pipe_preparacion_df.pickle'

ruta_pipe = ruta_proyecto + '/04_Modelos/' + nombre_pipe

# Guardado manual
with open(ruta_pipe, mode='wb') as file:
   cloudpickle.dump(pipe_preparacion_df, file)

In [10]:
df

,id,name,host_id,host_since,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms,beds,amenities,price,minimum_nights,maximum_nights,has_availability,availability_30,availability_60,availability_90,availability_365,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month,neighbourhood_group,precio_m2
0,21853,Bright and airy ...,83531,2010-02-21,None,None,0%,f,2.00,2.00,"['email', 'phone']",t,t,Cármenes,40.40,-3.74,Private room in ...,Private room,1,1.00,1 bath,1.00,1.00,"[""First aid kit""...",$31.00,4,40,t,0,0,0,144,33,0,0,2014-10-10,2018-07-15,4.58,4.72,4.56,4.75,4.82,4.21,4.67,None,f,2,0,2,0,0.27,Latina,2772
1,30320,Great Vacational...,130907,2010-05-24,None,None,None,f,3.00,6.00,"['email', 'phone']",t,f,Sol,40.41,-3.70,Entire rental unit,Entire home/apt,2,NaN,1 bath,1.00,NaN,"[""Heating"", ""Wif...",None,5,180,None,0,0,0,0,172,0,0,2010-07-06,2022-09-26,4.63,4.71,4.88,4.82,4.78,4.90,4.69,None,f,3,3,0,0,0.98,Centro,6170
2,30959,Beautiful loft i...,132883,2010-05-26,None,None,None,f,1.00,4.00,"['email', 'phone']",t,f,Embajadores,40.41,-3.70,Entire loft,Entire home/apt,2,NaN,1 bath,1.00,NaN,"[""Breakfast"", ""H...",None,3,730,None,0,0,0,0,8,0,0,2015-05-12,2017-05-30,4.38,4.14,4.38,4.63,4.63,4.88,4.25,None,f,1,1,0,0,0.07,Centro,6170
3,40916,Holiday Apartmen...,130907,2010-05-24,None,None,None,f,3.00,6.00,"['email', 'phone']",t,f,Universidad,40.42,-3.71,Entire rental unit,Entire home/apt,3,NaN,1 bath,1.00,NaN,"[""Heating"", ""Wif...",None,5,180,None,0,0,0,0,49,0,0,2010-11-01,2021-12-11,4.65,4.69,4.90,4.85,4.79,4.88,4.55,None,f,3,3,0,0,0.29,Centro,6170
4,62423,MAGIC ARTISTIC H...,303845,2010-11-29,within an hour,100%,100%,f,3.00,3.00,"['email', 'phone']",t,t,Justicia,40.42,-3.70,Private room in ...,Private room,4,1.50,1.5 shared baths,1.00,2.00,"[""Books and read...",$69.00,1,30,t,13,37,59,332,219,44,3,2018-05-10,2024-11-24,4.64,4.78,4.42,4.79,4.85,4.97,4.58,None,f,3,1,2,0,2.73,Centro,6170
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26755,1308816654087147793,Vive Madrid Río ...,251333188,2019-03-26,within a few hours,100%,67%,None,163.00,234.00,"['email', 'phone']",t,t,Puerta del Angel,40.41,-3.72,Entire rental unit,Entire home/apt,4,1.00,1 bath,1.00,2.00,"[""Wifi"", ""Kitche...",$258.00,1,365,t,15,45,75,350,0,0,0,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,t,19,19,0,0,NaN,Latina,2772
26756,1308818613904246041,Alquilo habitaci...,118339834,2017-02-27,None,None,None,f,1.00,1.00,"['email', 'phone']",t,t,Palacio,40.41,-3.71,Private room in ...,Private room,1,1.00,1 shared bath,1.00,1.00,"[""Wifi"", ""Kitche...",$20.00,15,16,t,29,59,89,364,0,0,0,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,f,1,0,1,0,NaN,Centro,6170
26757,1308839477505592914,Sierra Habitacio...,23441165,2014-11-07,within an hour,95%,98%,f,39.00,90.00,"['email', 'phone']",t,t,Numancia,40.40,-3.66,Room in hotel,Private room,2,1.00,1 private bath,1.00,1.00,"[""Exterior secur...",$69.00,1,365,t,14,44,74,309,0,0,0,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,t,26,0,23,0,NaN,Puente de Vallecas,2367
26758,1308842708819314249,Estudio luminoso...,251333188,2019-03-26,within a few hours,100%,67%,None,163.00,234.00,"['email', 'phone']",t,t,Puerta del Angel,40.41,-3.72,Entire rental

## CREAR LA MATRIZ DE PROCESOS

## CREAR EL PIPELINE modificaciones_totales

### FunctionTransformer: calidad_datos

In [12]:
#Traemos las variables que corresponden. Todas las funciones que usamos anteriormente en los notebooks están aquí 
pd.set_option('display.max_columns', None)

def calidad_datos(df):
    
    # Limpiamos los nombres de las variables por si hubiera algun carácter extraño
    df = clean_names(df)
    
    # Borramos duplicados
    #df.drop_duplicates(inplace = True)
    
    # Creamos una tabla temporal con una copia de df para trabajar sobre ella
    temp = df.copy()
    
    #PRICE
    def price(temp):
        
        # Asegurarse de que 'price' es numérico
        temp['price'] = temp['price'].replace(r'[\$,]', '', regex=True).astype(float)

        # 1. Promedio por 'neighbourhood_cleansed' y 'accommodates'
        grouped_price = temp.groupby(['neighbourhood_cleansed', 'accommodates'])['price'].mean()

        # 2. Promedio por 'accommodates'
        grouped_accommodates_price = temp.groupby('accommodates')['price'].mean()

        # 3. Promedio general
        global_price_mean = temp['price'].mean()

        # Función de imputación
        def imputar_precio(registro):
            if pd.isna(registro['price']):
                val = grouped_price.get((registro['neighbourhood_cleansed'], registro['accommodates']), np.nan)
                if pd.isna(val):
                    val = grouped_accommodates_price.get(registro['accommodates'], np.nan)
                if pd.isna(val):
                    val = global_price_mean
                return round(val) if pd.notna(val) else np.nan
            else:
                return registro['price']

        # Aplicar la imputación
        temp['price'] = temp.apply(imputar_precio, axis=1)   
        return temp        
    temp = price(temp)
    
    # Seleccionar solamente los inmuebles con un precio superior a 20€ por noche
    temp = temp.loc[temp['price']>=20]
    
    #BEDS
    def imputar_beds(registro):
        if pd.notna(registro['beds']):
            return registro['beds']
        if registro['accommodates'] <= 2:
            return 1
        elif registro['accommodates'] <= 4:
            return 2
        elif registro['accommodates'] <= 6:
            return 3
        else:
            return 4
        
    temp['beds'] = temp.apply(imputar_beds, axis=1).astype(float) 
    
    #BEDROOMS
    def imputar_bedrooms(registro):
        if pd.notna(registro['bedrooms']):
            return registro['bedrooms']
        if registro['beds'] <= 2:
            return 1
        elif registro['beds'] <= 4:
            return 2
        elif registro['beds'] <= 6:
            return 3
        else:
            return 4
        
    temp['bedrooms'] = temp.apply(imputar_bedrooms, axis=1).astype(float) 
    
    #BATHROOMS
    '''
    1. bathrooms_text None --> 0
    2. bathrooms_text quitamos el texto detras del numero
    3. bathrooms_text sustituimos los NaN generados por 0.5
    4. bathrooms sustituimos los NaN por los valores de bathrooms_text del mismo registro
    5. Eliminamos bathrooms_text porque ya tenemos completa la variable bathrooms
    '''  
    def bathrooms(temp):
        temp['bathrooms_text'] = temp['bathrooms_text']\
            .replace(np.nan, '0')\
            .replace(r'[^\d.]', '', regex=True)\
            .replace('', '0.5').astype('float64')
        temp['bathrooms'] = np.where(temp.bathrooms.isna(), temp.bathrooms_text, temp.bathrooms)
        #temp.drop(columns='bathrooms_text', inplace=True)
        return temp
    temp = bathrooms(temp)      
    
    #REVIEWS
    var_reviews = ['review_scores_rating',
                     'review_scores_accuracy',
                     'review_scores_cleanliness',
                     'review_scores_checkin',
                     'review_scores_communication',
                     'review_scores_location',
                     'review_scores_value']
    valor = 0
    def reviews(temp):
        for col in var_reviews:
            temp[col] = np.where(
                temp[col].isna(), 
                temp['review_scores_rating'], 
                temp[col])
        temp[var_reviews] = temp[var_reviews].fillna(valor)
        return temp
    temp = reviews(temp)
    
    #HOST RESPONSE TIME    
    valor = 'no response'
    temp['host_response_time'] = temp['host_response_time'].fillna(valor)
    
    #HOST RESPONSE RATE
    temp['host_response_rate'] = temp['host_response_rate'].replace('%','', regex=True)
    temp['host_response_rate'] = np.where(temp['host_response_rate'].isna(), 0,\
                                          temp['host_response_rate']).astype('float')
    
    #LICENSE
    temp['license'] = np.where(temp['license'].notna(), 'yes', 'no')    
    
    #HAS AVAILABILITY
    temp['has_availability'] = np.where(temp['has_availability'].notna(), 'yes', 'no')

    #HOST IS SUPERHOST
    def host_is_superhost(temp):
        # Función para imputar
        def imputar_is_superhost(registro):
            if pd.isna(registro['host_is_superhost']):
                scores = [
                    registro['review_scores_cleanliness'],
                    registro['review_scores_accuracy'],
                    registro['review_scores_rating'],
                    registro['review_scores_communication'],
                    registro['review_scores_checkin'],
                    registro['review_scores_location'],
                    registro['review_scores_value']
                ]
                # Verificar que todos los puntajes no son nulos y mayores a 4.5
                if all(pd.notna(score) and score > 4.5 for score in scores):
                    return 't'
                else:
                    return 'f'
            return registro['host_is_superhost']

        # Aplicar la imputación
        temp['host_is_superhost'] = temp.apply(imputar_is_superhost, axis=1)

        return temp
    temp = host_is_superhost(temp)
    
    #HOST VERIFICATIONS
    temp['host_verifications'] = np.where(temp['host_verifications']=='[]', "['No']", temp['host_verifications'])
    temp['host_verifications'] = np.where(temp.host_verifications == "['email', 'work_email']", 
                                          "['email']", 
                                          temp.host_verifications)
    
    #MINIMUM NIGHTS / MAXIMUM NIGHTS
    def atipicos_desv_tip(variable, num_desv_tip = 4):
        #sacamos los nulos por ahora
        variable = variable.dropna()
        #calculamos los límites
        media = np.mean(variable)
        sd = np.std(variable)
        umbral = sd * num_desv_tip
        lim_inf = media - umbral
        lim_sup = media + umbral
        #encontramos los índices de los que están fuera de los límites
        indices = [indice for indice,valor in variable.items() if valor < lim_inf or valor > lim_sup]
        return(indices)
    var_atipicos_desv_tip = ['minimum_nights', 'maximum_nights']
 
    # ELIMINAMOS LOS HOTELES PORQUE NO SON NUESTRO OBJETIVO
    temp = temp.loc[~((temp.property_type.str.contains('hotel', case=False, na=False))|
                      (temp.property_type.str.contains('hostel', case=False, na=False))|
                      (temp.room_type.str.contains('hotel', case=False, na=False)))]
    
    #Agrupar categorias raras
    def agrupar_cat_raras(variable, criterio = 0.02):
        #Calcula las frecuencias
        frecuencias = variable.value_counts(normalize=True)
        #Identifica las que están por debajo del criterio
        temp = [cada for cada in frecuencias.loc[frecuencias < criterio].index.values]
        #Las recodifica en 'OTROS'
        temp2 = np.where(variable.isin(temp),'OTROS',variable)
        #Devuelve el resultado
        return(temp2)
    var_agrupar_cat_raras = ['property_type']
    for variable in var_agrupar_cat_raras:
        temp[variable] = agrupar_cat_raras(temp[variable],criterio = 0.02)    
    
    # Eliminamos todas los registros que no tengan 50 variables sin nulos después de todo el proceso de calidad de datos 
    criterio = 50 #Elimina todos los registros que no tengan el número de variables del criterio sin nulos
    temp.dropna(thresh=criterio, inplace=True)
    temp.isna().sum().sort_values(ascending=True)
    
    return temp
df = calidad_datos(df)
df    

,id,name,host_id,host_since,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms,beds,amenities,price,minimum_nights,maximum_nights,has_availability,availability_30,availability_60,availability_90,availability_365,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month,neighbourhood_group,precio_m2
0,21853,Bright and airy ...,83531,2010-02-21,no response,0.00,0%,f,2.00,2.00,"['email', 'phone']",t,t,Cármenes,40.40,-3.74,Private room in ...,Private room,1,1.00,1.00,1.00,1.00,"[""First aid kit""...",31.00,4,40,yes,0,0,0,144,33,0,0,2014-10-10,2018-07-15,4.58,4.72,4.56,4.75,4.82,4.21,4.67,no,f,2,0,2,0,0.27,Latina,2772
1,30320,Great Vacational...,130907,2010-05-24,no response,0.00,None,f,3.00,6.00,"['email', 'phone']",t,f,Sol,40.41,-3.70,Entire rental unit,Entire home/apt,2,1.00,1.00,1.00,1.00,"[""Heating"", ""Wif...",149.00,5,180,no,0,0,0,0,172,0,0,2010-07-06,2022-09-26,4.63,4.71,4.88,4.82,4.78,4.90,4.69,no,f,3,3,0,0,0.98,Centro,6170
2,30959,Beautiful loft i...,132883,2010-05-26,no response,0.00,None,f,1.00,4.00,"['email', 'phone']",t,f,Embajadores,40.41,-3.70,Entire loft,Entire home/apt,2,1.00,1.00,1.00,1.00,"[""Breakfast"", ""H...",92.00,3,730,no,0,0,0,0,8,0,0,2015-05-12,2017-05-30,4.38,4.14,4.38,4.63,4.63,4.88,4.25,no,f,1,1,0,0,0.07,Centro,6170
3,40916,Holiday Apartmen...,130907,2010-05-24,no response,0.00,None,f,3.00,6.00,"['email', 'phone']",t,f,Universidad,40.42,-3.71,Entire rental unit,Entire home/apt,3,1.00,1.00,1.00,2.00,"[""Heating"", ""Wif...",124.00,5,180,no,0,0,0,0,49,0,0,2010-11-01,2021-12-11,4.65,4.69,4.90,4.85,4.79,4.88,4.55,no,f,3,3,0,0,0.29,Centro,6170
4,62423,MAGIC ARTISTIC H...,303845,2010-11-29,within an hour,100.00,100%,f,3.00,3.00,"['email', 'phone']",t,t,Justicia,40.42,-3.70,Private room in ...,Private room,4,1.50,1.50,1.00,2.00,"[""Books and read...",69.00,1,30,yes,13,37,59,332,219,44,3,2018-05-10,2024-11-24,4.64,4.78,4.42,4.79,4.85,4.97,4.58,no,f,3,1,2,0,2.73,Centro,6170
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26753,1308797105765566482,Elegancia y Como...,251333188,2019-03-26,within a few hours,100.00,67%,f,163.00,234.00,"['email', 'phone']",t,t,Casa de Campo,40.42,-3.72,Entire rental unit,Entire home/apt,2,1.00,1.00,1.00,1.00,"[""Wifi"", ""Kitche...",258.00,1,365,yes,30,60,90,365,0,0,0,None,None,0.00,0.00,0.00,0.00,0.00,0.00,0.00,no,t,19,19,0,0,NaN,Moncloa,4876
26754,1308811268356474614,Habitación tranq...,548687839,2023-11-29,within an hour,100.00,69%,f,1.00,2.00,['phone'],t,t,Ventas,40.43,-3.65,Private room in ...,Private room,2,1.00,1.00,1.00,1.00,"[""Wifi"", ""TV""]",85.00,5,15,yes,30,60,90,365,0,0,0,None,None,0.00,0.00,0.00,0.00,0.00,0.00,0.00,no,f,1,0,1,0,NaN,Ciudad Lineal,3676
26755,1308816654087147793,Vive Madrid Río ...,251333188,2019-03-26,within a few hours,100.00,67%,f,163.00,234.00,"['email', 'phone']",t,t,Puerta del Angel,40.41,-3.72,Entire rental unit,Entire home/apt,4,1.00,1.00,1.00,2.00,"[""Wifi"", ""Kitche...",258.00,1,365,yes,15,45,75,350,0,0,0,None,None,0.00,0.00,0.00,0.00,0.00,0.00,0.00,no,t,19,19,0,0,NaN,Latina,2772
26758,1308842708819314249,Estudio luminoso...,251333188,2019-03-26,within a few hours,100.00,67%,f,163.00,234.00,"['email', 'phone']",t,t,Puerta del Angel,40.41,-

In [13]:
df.isna().sum().sort_values(ascending=False)

reviews_per_month                               3408
first_review                                    3408
last_review                                     3408
host_acceptance_rate                            2237
id                                                 0
review_scores_cleanliness                          0
availability_60                                    0
availability_90                                    0
availability_365                                   0
number_of_reviews                                  0
number_of_reviews_ltm                              0
number_of_reviews_l30d                             0
review_scores_rating                               0
review_scores_accuracy                             0
review_scores_communication                        0
review_scores_checkin                              0
has_availability                                   0
review_scores_location                             0
review_scores_value                           

#### Crear un FunctionTransformer con las modificaciones de calidad_datos

In [14]:
#FunctionTransformer con las modificaciones realizadas
calidad_datos_ft = FunctionTransformer(calidad_datos)

#Pipeline con las modificaciones hechas
pipe_calidad_datos = make_pipeline(calidad_datos_ft)

nombre_pipe = 'pipe_calidad_datos.pickle'

ruta_pipe = ruta_proyecto + '/04_Modelos/' + nombre_pipe

# Guardado manual
with open(ruta_pipe, mode='wb') as file:
   cloudpickle.dump(pipe_calidad_datos, file)

### FunctionTransformer: gestion_variables

In [15]:
pd.set_option('display.max_columns', None)
def gestion_variables(df):
    # 1. Estimación de m2 por número de dormitorios
    condiciones = [df.bedrooms == 0,
                   df.bedrooms == 1,
                   df.bedrooms == 2,
                   df.bedrooms == 3,
                   df.bedrooms == 4,
                   df.bedrooms > 4]
    resultados = [40, 50, 70, 90, 120, 150]
    df['m2'] = np.select(condiciones, resultados, default=-999)

    # 2. Estimación de precio de compra
    df['precio_compra'] = df['m2'] * df['precio_m2'] * 0.8

    # 3. Cálculo de disponibilidad promedio por tipo de habitación
    dispo_private = df[df['room_type'] == 'Private room']['availability_365'].mean() / 365
    dispo_shared = df[df['room_type'] == 'Shared room']['availability_365'].mean() / 365

    # 4. Cálculo del precio total estimado
    def crear_precio_total(registro):
        try:
            price = float(registro.price)
            beds = float(registro.beds)
        except (ValueError, TypeError):
            return np.nan

        room = registro.room_type
        if pd.isna(price) or pd.isna(beds) or pd.isna(room):
            return np.nan

        if beds > 1 and room == 'Private room':
            return price * beds * (1 - dispo_private)
        elif beds > 1 and room == 'Shared room':
            return price * beds * (1 - dispo_shared)
        else:
            return price

    df['precio_total'] = df.apply(crear_precio_total, axis=1).round(2)
    
    # 5. Determinar la ocupación del inmueble
    df['ocupacion'] = ((365 - df.availability_365) / 365 * 100).astype('int')
    
    # 6. Creación de las distancias a un pdi
    from math import radians, cos, sin, asin, sqrt
    def haversine(lat1, lon1, lat2, lon2):

          R = 6372.8 #En km, si usas millas tienes que cambiarlo por 3959.87433

          dLat = radians(lat2 - lat1)
          dLon = radians(lon2 - lon1)
          lat1 = radians(lat1)
          lat2 = radians(lat2)

          a = sin(dLat/2)**2 + cos(lat1)*cos(lat2)*sin(dLon/2)**2
          c = 2*asin(sqrt(a))

          return R * c
    #Las coordenadas de la Puerta del Sol serán lat1 y lon1
    lat1 = 40.4167278
    lon1 = -3.7033387

    df['pdi_sol'] = df.apply(lambda registro: haversine(lat1,lon1,registro['latitude'],registro['longitude']),axis=1)

    # 7. Determinar si el piso es rentable
    df['pisos_rentables'] = np.where(df['precio_compra'] / 10 > df['precio_total'] * df['ocupacion'] * 10, 0, 1)
    
    def variables_texto(df):
        path_vectorizer = os.path.join(ruta_proyecto, '04_Modelos', 'count_vectorizer.pkl')
        
        if os.path.exists(path_vectorizer):
            with open(path_vectorizer, "rb") as f:
                cv = pickle.load(f)
        else:
            cv = CountVectorizer()
            cv.fit(df["amenities"].astype(str))
            with open(path_vectorizer, "wb") as f:
                pickle.dump(cv, f)
        #Aplicamos
        caracteristicas = cv.transform(df["amenities"].astype(str))
        
        #Definimos un dataframe con los términos encontrados transformados a variables que mapean las columnas donde se encuentran
        terminos = pd.DataFrame(caracteristicas.toarray(), columns=cv.get_feature_names_out(), index=df.index)
        
        #Concatenar al DataFrame original        
        df = pd.concat([df, terminos], axis=1)    
        return df
    df = variables_texto(df)
    
    return df
df = gestion_variables(df)
df

,id,name,host_id,host_since,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms,beds,amenities,price,minimum_nights,maximum_nights,has_availability,availability_30,availability_60,availability_90,availability_365,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month,neighbourhood_group,precio_m2,m2,precio_compra,precio_total,ocupacion,pdi_sol,pisos_rentables,air,air conditioning,allowed,and silverware,basics,bed,bed linens,body,body soap,cleaning,clothing,clothing storage,coffee,coffee maker,conditioning,cooking,cooking basics,dedicated,dedicated workspace,dishes,dishes and,dishes and silverware,dryer bed,dryer bed linens,elevator,freezer,hair,hair dryer,hair dryer bed,hot water iron,in,iron,kitchen essentials,linens,maker,microwave,microwave hangers,oven,parking,refrigerator,room,shampoo,silverware,soap,storage,stove,tv washer,water iron,wifi kitchen,workspace
0,21853,Bright and airy ...,83531,2010-02-21,no response,0.00,0%,f,2.00,2.00,"['email', 'phone']",t,t,Cármenes,40.40,-3.74,Private room in ...,Private room,1,1.00,1.00,1.00,1.00,"[""First aid kit""...",31.00,4,40,yes,0,0,0,144,33,0,0,2014-10-10,2018-07-15,4.58,4.72,4.56,4.75,4.82,4.21,4.67,no,f,2,0,2,0,0.27,Latina,2772,50,110880.00,31.00,60,3.52,1,1,1,0,1,1,1,1,0,0,0,0,0,1,1,1,1,1,0,0,1,1,1,1,1,1,0,1,1,1,1,0,1,1,1,1,1,1,1,1,1,0,1,1,0,0,0,1,1,1,0
1,30320,Great Vacational...,130907,2010-05-24,no response,0.00,None,f,3.00,6.00,"['email', 'phone']",t,f,Sol,40.41,-3.70,Entire rental unit,Entire home/apt,2,1.00,1.00,1.00,1.00,"[""Heating"", ""Wif...",149.00,5,180,no,0,0,0,0,172,0,0,2010-07-06,2022-09-26,4.63,4.71,4.88,4.82,4.78,4.90,4.69,no,f,3,3,0,0,0.98,Centro,6170,50,246800.00,149.00,100,0.23,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,30959,Beautiful loft i...,132883,2010-05-26,no response,0.00,None,f,1.00,4.00,"['email', 'phone']",t,f,Embajadores,40.41,-3.70,Entire loft,Entire home/apt,2,1.00,1.00,1.00,1.00,"[""Breakfast"", ""H...",92.00,3,730,no,0,0,0,0,8,0,0,2015-05-12,2017-05-30,4.38,4.14,4.38,4.63,4.63,4.88,4.25,no,f,1,1,0,0,0.07,Centro,6170,50,246800.00,92.00,100,0.50,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
3,40916,Holiday Apartmen...,130907,2010-05-24,no response,0.00,None,f,3.00,6.00,"['email', 'phone']",t,f,Universidad,40.42,-3.71,Entire rental unit,Entire home/apt,3,1.00,1.00,1.00,2.00,"[""Heating"", ""Wif...",124.00,5,180,no,0,0,0,0,49,0,0,2010-11-01,2021-12-11,4.65,4.69,4.90,4.85,4.79,4.88,4.55,no,f,3,3,0,0,0.29,Centro,6170,50,246800.00,124.00,100,0.67,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0
4,62423,MAGIC ARTISTIC H...,303845,2010-11-29,within an hour,100.00,100%,f,3.00,3.00,"['email', 'phone']",t,t,Justicia,40.42,-3.70,Private room in ...,Private room,4,1.50,1.50,1.00,2.00,"[""Books and read...",69.00,1,30,yes,13,37,59,332,219,44,3,2018-05-10,2024-11-24,4.64,4.78,4.42,4.79,4.85,4.97,4.58,no,f,3,1,2,0,2.73,Centro,6170,50,246800.00,80.92,9,0.62,0,0,0,1,1,1,0,0,1,1,1,0,0,0,0,0,1,1,1,1,1,1,1,0,0,0,0,1,1,0,1,0,1,1,0,0,0,0,0,0,1,1,1,1,1,0,0,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,.

In [16]:
df

,id,name,host_id,host_since,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms,beds,amenities,price,minimum_nights,maximum_nights,has_availability,availability_30,availability_60,availability_90,availability_365,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month,neighbourhood_group,precio_m2,m2,precio_compra,precio_total,ocupacion,pdi_sol,pisos_rentables,air,air conditioning,allowed,and silverware,basics,bed,bed linens,body,body soap,cleaning,clothing,clothing storage,coffee,coffee maker,conditioning,cooking,cooking basics,dedicated,dedicated workspace,dishes,dishes and,dishes and silverware,dryer bed,dryer bed linens,elevator,freezer,hair,hair dryer,hair dryer bed,hot water iron,in,iron,kitchen essentials,linens,maker,microwave,microwave hangers,oven,parking,refrigerator,room,shampoo,silverware,soap,storage,stove,tv washer,water iron,wifi kitchen,workspace
0,21853,Bright and airy ...,83531,2010-02-21,no response,0.00,0%,f,2.00,2.00,"['email', 'phone']",t,t,Cármenes,40.40,-3.74,Private room in ...,Private room,1,1.00,1.00,1.00,1.00,"[""First aid kit""...",31.00,4,40,yes,0,0,0,144,33,0,0,2014-10-10,2018-07-15,4.58,4.72,4.56,4.75,4.82,4.21,4.67,no,f,2,0,2,0,0.27,Latina,2772,50,110880.00,31.00,60,3.52,1,1,1,0,1,1,1,1,0,0,0,0,0,1,1,1,1,1,0,0,1,1,1,1,1,1,0,1,1,1,1,0,1,1,1,1,1,1,1,1,1,0,1,1,0,0,0,1,1,1,0
1,30320,Great Vacational...,130907,2010-05-24,no response,0.00,None,f,3.00,6.00,"['email', 'phone']",t,f,Sol,40.41,-3.70,Entire rental unit,Entire home/apt,2,1.00,1.00,1.00,1.00,"[""Heating"", ""Wif...",149.00,5,180,no,0,0,0,0,172,0,0,2010-07-06,2022-09-26,4.63,4.71,4.88,4.82,4.78,4.90,4.69,no,f,3,3,0,0,0.98,Centro,6170,50,246800.00,149.00,100,0.23,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,30959,Beautiful loft i...,132883,2010-05-26,no response,0.00,None,f,1.00,4.00,"['email', 'phone']",t,f,Embajadores,40.41,-3.70,Entire loft,Entire home/apt,2,1.00,1.00,1.00,1.00,"[""Breakfast"", ""H...",92.00,3,730,no,0,0,0,0,8,0,0,2015-05-12,2017-05-30,4.38,4.14,4.38,4.63,4.63,4.88,4.25,no,f,1,1,0,0,0.07,Centro,6170,50,246800.00,92.00,100,0.50,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
3,40916,Holiday Apartmen...,130907,2010-05-24,no response,0.00,None,f,3.00,6.00,"['email', 'phone']",t,f,Universidad,40.42,-3.71,Entire rental unit,Entire home/apt,3,1.00,1.00,1.00,2.00,"[""Heating"", ""Wif...",124.00,5,180,no,0,0,0,0,49,0,0,2010-11-01,2021-12-11,4.65,4.69,4.90,4.85,4.79,4.88,4.55,no,f,3,3,0,0,0.29,Centro,6170,50,246800.00,124.00,100,0.67,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0
4,62423,MAGIC ARTISTIC H...,303845,2010-11-29,within an hour,100.00,100%,f,3.00,3.00,"['email', 'phone']",t,t,Justicia,40.42,-3.70,Private room in ...,Private room,4,1.50,1.50,1.00,2.00,"[""Books and read...",69.00,1,30,yes,13,37,59,332,219,44,3,2018-05-10,2024-11-24,4.64,4.78,4.42,4.79,4.85,4.97,4.58,no,f,3,1,2,0,2.73,Centro,6170,50,246800.00,80.92,9,0.62,0,0,0,1,1,1,0,0,1,1,1,0,0,0,0,0,1,1,1,1,1,1,1,0,0,0,0,1,1,0,1,0,1,1,0,0,0,0,0,0,1,1,1,1,1,0,0,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,.

#### Crear un FunctionTransformer con las modificaciones de gestion_variables

In [18]:
#FunctionTransformer con las modificaciones realizadas
gestion_variables_ft = FunctionTransformer(gestion_variables)

#Pipeline con las modificaciones hechas
pipe_gestion_variables = make_pipeline(gestion_variables_ft)

nombre_pipe = 'pipe_gestion_variables.pickle'

ruta_pipe = ruta_proyecto + '/04_Modelos/' + nombre_pipe

# Guardado manual
with open(ruta_pipe, mode='wb') as file:
   cloudpickle.dump(pipe_gestion_variables, file)

### PIPELINE: modificaciones_totales

In [19]:
#Pipeline con las modificaciones realizadas
pipe_modificaciones_totales = make_pipeline(pipe_calidad_datos, pipe_gestion_variables)

nombre_pipe = 'pipe_modificaciones_totales.pickle'

ruta_pipe = ruta_proyecto + '/04_Modelos/' + nombre_pipe

# Guardado manual
with open(ruta_pipe, mode='wb') as file:
   cloudpickle.dump(pipe_modificaciones_totales, file)

### Seleccionar solo las variables finales

#### 2.1.1. Cargar la lista de variables finales

Estas son las variables finales que resultan predictoras según el SuperRanking

He hecho este paso al final porque era necesario cargar varios datasets, hacerles diferentes modificaciones, unirlos, generar variables y modificar otras hasta llegar amenities que debía separar el contenido en palabras y generar variables predictoras con ellas

In [20]:
nombre_variables_finales = ruta_proyecto + '/05_Resultados/' + 'variables_finales.pickle'
pd.read_pickle(nombre_variables_finales).sort_index().index.to_list()

['accommodates_yeo',
 'allowed',
 'and silverware',
 'availability_30_yeo',
 'availability_365_yeo',
 'availability_90_yeo',
 'basics',
 'bathrooms_disc_ds',
 'bed',
 'bedrooms_disc_ds',
 'beds_disc_ds',
 'body',
 'calculated_host_listings_count_entire_homes_disc_ds',
 'cleaning',
 'clothing',
 'clothing storage',
 'dryer bed',
 'hair',
 'hair dryer',
 'has_availability_oe',
 'host_identity_verified_oe',
 'host_response_rate_yeo',
 'host_response_time_oe',
 'host_verifications_oe',
 'instant_bookable_oe',
 'license_oe',
 'maker',
 'maximum_nights_disc_ds',
 'maximum_nights_yeo',
 'microwave hangers',
 'neighbourhood_group_Arganzuela',
 'neighbourhood_group_Carabanchel',
 'neighbourhood_group_Centro',
 'neighbourhood_group_Chamartín',
 'neighbourhood_group_Latina',
 'neighbourhood_group_Moncloa',
 'neighbourhood_group_Moratalaz',
 'neighbourhood_group_Puente de Vallecas',
 'neighbourhood_group_Salamanca',
 'neighbourhood_group_Usera',
 'neighbourhood_group_Villa de Vallecas',
 'neighbou

Además, debemos añadirle a las variables 

#### Apuntar la lista de variables finales sin extensiones (MANUALMENTE)

In [21]:
'''
#Estas son las variables utilizadas para hacer las modificaciones pero que después no son predictoras

 'review_scores_rating',
 'review_scores_accuracy',
 'review_scores_cleanliness',
 'review_scores_checkin',
 'review_scores_communication',
 'review_scores_location',
 'review_scores_value'
 
 '''

"\n#Estas son las variables utilizadas para hacer las modificaciones pero que después no son predictoras\n\n 'review_scores_rating',\n 'review_scores_accuracy',\n 'review_scores_cleanliness',\n 'review_scores_checkin',\n 'review_scores_communication',\n 'review_scores_location',\n 'review_scores_value'\n \n "

In [22]:
#Se cogen las variables originales porque las variables con extensiones se generan al pasar los procesos y al pasar el código
#a producción se encontrará con un dataset sin modificaciones al inicio

variables_finales = [
'accommodates',
'availability_30',
'availability_365',
'availability_90',
'bedrooms',
'bed',
'calculated_host_listings_count',
'has_availability',
'host_response_rate',
'host_response_time',
'instant_bookable',
'maximum_nights',
'neighbourhood_group',
'price',
'property_type',
'room_type',
'license',
'bathrooms',
'air',
'allowed',
'cleaning',
'hot water iron',
'iron',
'microwave',
'parking',
'refrigerator',
'shampoo',
'tv washer',
'wifi kitchen',
'neighbourhood_cleansed',
'and silverware',
'body',
'clothing',
'clothing storage',
'coffee',
'elevator',
'hair',
'host_identity_verified',
'host_verifications',
'kitchen essentials',
'allowed',
'basics',
'beds',
'dryer bed',
'hair dryer',
'maker',
'microwave hangers',
'oven',
'stove',
'review_scores_rating',
'review_scores_accuracy',
'review_scores_cleanliness',
'review_scores_checkin',
'review_scores_communication',
'review_scores_location',
'review_scores_value',
'host_is_superhost',
'bathrooms_text',
'precio_m2',
'latitude',
'longitude',
'amenities',
'number_of_reviews']





#### Para x

Quedarse solo con las de la lista.

In [23]:
x = df[variables_finales].copy() #Nos quedamos solamente las variables que hemos seleccionado
x.head(2)

,accommodates,availability_30,availability_365,availability_90,bedrooms,bed,calculated_host_listings_count,has_availability,host_response_rate,host_response_time,instant_bookable,maximum_nights,neighbourhood_group,price,property_type,room_type,license,bathrooms,air,allowed,cleaning,hot water iron,iron,microwave,parking,refrigerator,shampoo,tv washer,wifi kitchen,neighbourhood_cleansed,and silverware,body,clothing,clothing storage,coffee,elevator,hair,host_identity_verified,host_verifications,kitchen essentials,allowed,basics,beds,dryer bed,hair dryer,maker,microwave hangers,oven,stove,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,host_is_superhost,bathrooms_text,precio_m2,latitude,longitude,amenities,number_of_reviews
0,1,0,144,0,1.00,1,2,yes,0.00,no response,f,40,Latina,31.00,Private room in ...,Private room,no,1.00,1,0,0,1,1,1,1,1,1,1,1,Cármenes,1,0,0,0,1,1,1,t,"['email', 'phone']",1,0,1,1.00,1,1,1,1,1,0,4.58,4.72,4.56,4.75,4.82,4.21,4.67,f,1.00,2772,40.40,-3.74,"[""First aid kit""...",33
1,2,0,0,0,1.00,0,3,no,0.00,no response,f,180,Centro,149.00,Entire rental unit,Entire home/apt,no,1.00,1,0,0,0,0,0,0,0,0,0,0,Sol,0,0,0,0,0,1,0,f,"['email', 'phone']",0,0,0,1.00,0,0,0,0,0,0,4.63,4.71,4.88,4.82,4.78,4.90,4.69,f,1.00,6170,40.41,-3.70,"[""Heating"", ""Wif...",172


In [24]:
x

,accommodates,availability_30,availability_365,availability_90,bedrooms,bed,calculated_host_listings_count,has_availability,host_response_rate,host_response_time,instant_bookable,maximum_nights,neighbourhood_group,price,property_type,room_type,license,bathrooms,air,allowed,cleaning,hot water iron,iron,microwave,parking,refrigerator,shampoo,tv washer,wifi kitchen,neighbourhood_cleansed,and silverware,body,clothing,clothing storage,coffee,elevator,hair,host_identity_verified,host_verifications,kitchen essentials,allowed,basics,beds,dryer bed,hair dryer,maker,microwave hangers,oven,stove,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,host_is_superhost,bathrooms_text,precio_m2,latitude,longitude,amenities,number_of_reviews
0,1,0,144,0,1.00,1,2,yes,0.00,no response,f,40,Latina,31.00,Private room in ...,Private room,no,1.00,1,0,0,1,1,1,1,1,1,1,1,Cármenes,1,0,0,0,1,1,1,t,"['email', 'phone']",1,0,1,1.00,1,1,1,1,1,0,4.58,4.72,4.56,4.75,4.82,4.21,4.67,f,1.00,2772,40.40,-3.74,"[""First aid kit""...",33
1,2,0,0,0,1.00,0,3,no,0.00,no response,f,180,Centro,149.00,Entire rental unit,Entire home/apt,no,1.00,1,0,0,0,0,0,0,0,0,0,0,Sol,0,0,0,0,0,1,0,f,"['email', 'phone']",0,0,0,1.00,0,0,0,0,0,0,4.63,4.71,4.88,4.82,4.78,4.90,4.69,f,1.00,6170,40.41,-3.70,"[""Heating"", ""Wif...",172
2,2,0,0,0,1.00,0,1,no,0.00,no response,f,730,Centro,92.00,Entire loft,Entire home/apt,no,1.00,0,1,0,0,0,0,0,0,1,0,0,Embajadores,0,0,0,0,0,1,0,f,"['email', 'phone']",0,1,0,1.00,0,0,0,0,0,0,4.38,4.14,4.38,4.63,4.63,4.88,4.25,f,1.00,6170,40.41,-3.70,"[""Breakfast"", ""H...",8
3,3,0,0,0,1.00,0,3,no,0.00,no response,f,180,Centro,124.00,Entire rental unit,Entire home/apt,no,1.00,1,1,0,0,0,0,0,0,0,1,0,Universidad,0,0,0,0,0,1,0,f,"['email', 'phone']",0,1,0,2.00,0,0,0,0,0,0,4.65,4.69,4.90,4.85,4.79,4.88,4.55,f,1.00,6170,40.42,-3.71,"[""Heating"", ""Wif...",49
4,4,13,332,59,1.00,0,3,yes,100.00,within an hour,f,30,Centro,69.00,Private room in ...,Private room,no,1.50,0,1,1,1,1,0,0,1,1,0,0,Justicia,1,1,0,0,0,0,1,t,"['email', 'phone']",1,1,1,2.00,0,1,0,0,0,0,4.64,4.78,4.42,4.79,4.85,4.97,4.58,f,1.50,6170,40.42,-3.70,"[""Books and read...",219
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26753,2,30,365,90,1.00,1,19,yes,100.00,within a few hours,t,365,Moncloa,258.00,Entire rental unit,Entire home/apt,no,1.00,1,0,0,0,0,1,0,1,1,1,1,Casa de Campo,1,1,1,1,1,0,1,t,"['email', 'phone']",0,0,1,1.00,1,1,1,1,1,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,f,1.00,4876,40.42,-3.72,"[""Wifi"", ""Kitche...",0
26754,2,30,365,90,1.00,0,1,yes,100.00,within an hour,f,15,Ciudad Lineal,85.00,Private room in ...,Private room,no,1.00,0,0,0,0,0,0,0,0,0,0,0,Ventas,0,0,0,0,0,0,0,t,['phone'],0,0,0,1.00,0,0,0,0,0,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,f,1.00,3676,40.43,-3.65,"[""Wifi"", ""TV""]",0
26755,4,15,350,75,1.00,1,19,yes,100.00,within a few hours,t,365,Latina,258.00,Entire rental unit,Entire home/apt,no,1.00,1,0,0,0,0,1,0,1,1,1,1,Puerta del Angel,1,1,1,1,1,0,1,t,"['email', 'phone']",0,0,1,2.00,1,1,1,1,1,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,f,1.00,2772,40.41,-3.72,"[""Wifi"", ""Kitche...",0
26758,2,13,348,73,1.00,1,19,yes,100.00,within a few hours,t,365,Latina,258.00,Entire rental unit,Entire home/apt,no,1.00,1,0,0,0,0,1,0,1,1,1,1,Puerta del Angel,1,1,1,1,1,0,1,t,"['email', 'phone']",0,0,1,2.00,1,1,1,1,1,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,f,1.00,2772,40.41,-3.72,"[""Wifi"", ""Kitche...",0


#### Para y

Especificar la target con el nombre de la columna que es nuestra target

In [25]:
target = 'pisos_rentables'

Crear el y.

In [26]:
y = df[target].copy()

In [27]:
print(x.shape, y.shape)

(24079, 63) (24079,)


In [28]:
pd.set_option('display.max_rows', 100)
x.isna().sum().sort_values(ascending=False)

accommodates                      0
oven                              0
coffee                            0
elevator                          0
hair                              0
host_identity_verified            0
host_verifications                0
kitchen essentials                0
allowed                           0
basics                            0
beds                              0
dryer bed                         0
hair dryer                        0
maker                             0
microwave hangers                 0
stove                             0
clothing                          0
review_scores_rating              0
review_scores_accuracy            0
review_scores_cleanliness         0
review_scores_checkin             0
review_scores_communication       0
review_scores_location            0
review_scores_value               0
host_is_superhost                 0
bathrooms_text                    0
precio_m2                         0
latitude                    

In [29]:
y

0        1
1        1
2        1
3        1
4        0
        ..
26753    0
26754    0
26755    0
26758    0
26759    1
Name: pisos_rentables, Length: 24079, dtype: int32

In [30]:
x

,accommodates,availability_30,availability_365,availability_90,bedrooms,bed,calculated_host_listings_count,has_availability,host_response_rate,host_response_time,instant_bookable,maximum_nights,neighbourhood_group,price,property_type,room_type,license,bathrooms,air,allowed,cleaning,hot water iron,iron,microwave,parking,refrigerator,shampoo,tv washer,wifi kitchen,neighbourhood_cleansed,and silverware,body,clothing,clothing storage,coffee,elevator,hair,host_identity_verified,host_verifications,kitchen essentials,allowed,basics,beds,dryer bed,hair dryer,maker,microwave hangers,oven,stove,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,host_is_superhost,bathrooms_text,precio_m2,latitude,longitude,amenities,number_of_reviews
0,1,0,144,0,1.00,1,2,yes,0.00,no response,f,40,Latina,31.00,Private room in ...,Private room,no,1.00,1,0,0,1,1,1,1,1,1,1,1,Cármenes,1,0,0,0,1,1,1,t,"['email', 'phone']",1,0,1,1.00,1,1,1,1,1,0,4.58,4.72,4.56,4.75,4.82,4.21,4.67,f,1.00,2772,40.40,-3.74,"[""First aid kit""...",33
1,2,0,0,0,1.00,0,3,no,0.00,no response,f,180,Centro,149.00,Entire rental unit,Entire home/apt,no,1.00,1,0,0,0,0,0,0,0,0,0,0,Sol,0,0,0,0,0,1,0,f,"['email', 'phone']",0,0,0,1.00,0,0,0,0,0,0,4.63,4.71,4.88,4.82,4.78,4.90,4.69,f,1.00,6170,40.41,-3.70,"[""Heating"", ""Wif...",172
2,2,0,0,0,1.00,0,1,no,0.00,no response,f,730,Centro,92.00,Entire loft,Entire home/apt,no,1.00,0,1,0,0,0,0,0,0,1,0,0,Embajadores,0,0,0,0,0,1,0,f,"['email', 'phone']",0,1,0,1.00,0,0,0,0,0,0,4.38,4.14,4.38,4.63,4.63,4.88,4.25,f,1.00,6170,40.41,-3.70,"[""Breakfast"", ""H...",8
3,3,0,0,0,1.00,0,3,no,0.00,no response,f,180,Centro,124.00,Entire rental unit,Entire home/apt,no,1.00,1,1,0,0,0,0,0,0,0,1,0,Universidad,0,0,0,0,0,1,0,f,"['email', 'phone']",0,1,0,2.00,0,0,0,0,0,0,4.65,4.69,4.90,4.85,4.79,4.88,4.55,f,1.00,6170,40.42,-3.71,"[""Heating"", ""Wif...",49
4,4,13,332,59,1.00,0,3,yes,100.00,within an hour,f,30,Centro,69.00,Private room in ...,Private room,no,1.50,0,1,1,1,1,0,0,1,1,0,0,Justicia,1,1,0,0,0,0,1,t,"['email', 'phone']",1,1,1,2.00,0,1,0,0,0,0,4.64,4.78,4.42,4.79,4.85,4.97,4.58,f,1.50,6170,40.42,-3.70,"[""Books and read...",219
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26753,2,30,365,90,1.00,1,19,yes,100.00,within a few hours,t,365,Moncloa,258.00,Entire rental unit,Entire home/apt,no,1.00,1,0,0,0,0,1,0,1,1,1,1,Casa de Campo,1,1,1,1,1,0,1,t,"['email', 'phone']",0,0,1,1.00,1,1,1,1,1,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,f,1.00,4876,40.42,-3.72,"[""Wifi"", ""Kitche...",0
26754,2,30,365,90,1.00,0,1,yes,100.00,within an hour,f,15,Ciudad Lineal,85.00,Private room in ...,Private room,no,1.00,0,0,0,0,0,0,0,0,0,0,0,Ventas,0,0,0,0,0,0,0,t,['phone'],0,0,0,1.00,0,0,0,0,0,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,f,1.00,3676,40.43,-3.65,"[""Wifi"", ""TV""]",0
26755,4,15,350,75,1.00,1,19,yes,100.00,within a few hours,t,365,Latina,258.00,Entire rental unit,Entire home/apt,no,1.00,1,0,0,0,0,1,0,1,1,1,1,Puerta del Angel,1,1,1,1,1,0,1,t,"['email', 'phone']",0,0,1,2.00,1,1,1,1,1,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,f,1.00,2772,40.41,-3.72,"[""Wifi"", ""Kitche...",0
26758,2,13,348,73,1.00,1,19,yes,100.00,within a few hours,t,365,Latina,258.00,Entire rental unit,Entire home/apt,no,1.00,1,0,0,0,0,1,0,1,1,1,1,Puerta del Angel,1,1,1,1,1,0,1,t,"['email', 'phone']",0,0,1,2.00,1,1,1,1,1,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,f,1.00,2772,40.41,-3.72,"[""Wifi"", ""Kitche...",0


### 4.2. Instanciar transformación de variables

In [31]:
#ENCODING
var_ohe = ['property_type',
           'room_type',
           'neighbourhood_group']
ohe = OneHotEncoder(sparse_output = False, handle_unknown='ignore')


var_oe = ['host_response_time',
          'host_is_superhost',
          'host_identity_verified',
          'has_availability',
          'license',
          'instant_bookable']
#Orden de las variables
orden_host_response_time = ['no response', 'a few days or more','within a day', 'within a few hours', 'within an hour' ]
orden_host_is_superhost = ['f','t']
orden_host_identity_verified = ['f','t']
orden_has_availability = ['yes','no']
orden_license = ['no','yes']
orden_instant_bookable = ['f','t']
ordinal_scaled_pipeline = Pipeline([
    ('oe', OrdinalEncoder(categories=[orden_host_response_time,
                                      orden_host_is_superhost,
                                      orden_host_identity_verified,
                                      orden_has_availability,
                                      orden_license,
                                      orden_instant_bookable],
                          handle_unknown='use_encoded_value',
                          unknown_value=10)),
    ('mms', MinMaxScaler())
])

#DISCRETIZACIÓN
var_disc_ds = ['price',
 'maximum_nights',
 'calculated_host_listings_count',
 'accommodates',
 'beds',
 'bedrooms',
 'number_of_reviews']
disc_ds = KBinsDiscretizer(n_bins = 6, strategy = 'uniform', encode = 'ordinal')

#var_disc_cs = []
#disc_cs = KBinsDiscretizer(n_bins = 2, strategy = 'quantile', encode = 'ordinal')

#BINARIZACIÓN
#var_bin = []
#bin = Binarizer(threshold=0)

#NORMALIZACIÓN GAUSS
var_yeo = ['host_response_rate',
           'accommodates',
           'maximum_nights',
           'price',
           'availability_30',
           'availability_90',
           'availability_365']
yeo = PowerTransformer(method = 'yeo-johnson')

#var_qt = []
#qt = QuantileTransformer(output_distribution='normal')

#REESCALADO
var_mms = [#'host_response_time',
 #'host_is_superhost',
 #'host_identity_verified',
 #'has_availability',
 #'license',
 #'instant_bookable',
 'price',
 'maximum_nights',
 'calculated_host_listings_count',
 'beds',
 'bedrooms',
 'number_of_reviews',
 'host_response_rate',
 'accommodates',
 'availability_30',
 'availability_90',
 'availability_365']
mms = MinMaxScaler()

#var_ss = []
#ss = StandardScaler()

#var_rs = []
#p_min=10
#p_max=90
#rs = RobustScaler(quantile_range=(p_min, p_max))

### 4.3. Crear el pipe del preprocesamiento

#### 4.3.1. Crear el column transformer

In [32]:
ct = make_column_transformer(
    (ohe, var_ohe),
    (ordinal_scaled_pipeline, var_oe),  
    (disc_ds, var_disc_ds),
#    (disc_cs, var_disc_cs),
#    (bin, var_bin),
    (yeo, var_yeo),    
#    (qt, var_qt),
    (mms, [col for col in var_mms if col not in var_oe]),#    (ss, var_ss),
#    (rs, var_rs),
    remainder='drop') #En este caso no hay mas variables porque las hemos usado todas. Podría ser: 'drop' o 'passthrough'

#### 4.3.2. Crear el pipeline del preprocesamiento

In [33]:
pipe_prepro = make_pipeline(pipe_modificaciones_totales, ct)

In [34]:
# Debug del paso de transformación
#x_transformado = pipe_prepro.fit_transform(x)
#print(x_transformado.shape)


### 4.4. Instanciar el modelo

#### 4.4.1. Instanciar el algoritmo 

El que tuviera el best_estimator_ de Modelizacion para Clasificacion/Regresión

In [35]:
print(y.isnull().sum())  # Debe ser 0

# Si hay nulos, puedes hacer esto:
mask = ~y.isnull()
x = x.loc[mask]
y = y.loc[mask]


0


In [36]:
modelo = XGBClassifier(learning_rate= 0.05,
                       max_depth= 5,
                       n_estimators= 1000,
                       n_jobs= -1,
                       reg_alpha= 0.1,
                       reg_lambda= 1,
                       verbosity= 0)

#### 4.4.2. Crear el pipe final de entrenamiento

In [37]:
pipe_entrenamiento = make_pipeline(pipe_prepro,modelo)

#### 4.4.3. Guardar el pipe final de entrenamiento

In [38]:
nombre_pipe = 'pipe_entrenamiento.pickle'

ruta_pipe = ruta_proyecto + '/04_Modelos/' + nombre_pipe

# Guardado manual
with open(ruta_pipe, mode='wb') as file:
   cloudpickle.dump(pipe_entrenamiento, file)

#### 4.4.4. Entrenar el pipe final de ejecución

In [39]:
pipe_ejecucion = pipe_entrenamiento.fit(x,y)

In [40]:
print(x.shape)
print(y.shape)

(24079, 63)
(24079,)


In [41]:
print(x.index.equals(y.index))  # Debe dar True

True


In [42]:
x.isna().sum().sort_values(ascending=False)

accommodates                      0
oven                              0
coffee                            0
elevator                          0
hair                              0
host_identity_verified            0
host_verifications                0
kitchen essentials                0
allowed                           0
basics                            0
beds                              0
dryer bed                         0
hair dryer                        0
maker                             0
microwave hangers                 0
stove                             0
clothing                          0
review_scores_rating              0
review_scores_accuracy            0
review_scores_cleanliness         0
review_scores_checkin             0
review_scores_communication       0
review_scores_location            0
review_scores_value               0
host_is_superhost                 0
bathrooms_text                    0
precio_m2                         0
latitude                    

In [43]:
print(x.index.equals(y.index))  # Debe dar True


True


## 5. GUARDAR EL PIPE

### 5.1. Nombre del pipe final de ejecución

In [44]:
nombre_pipe = 'pipe_ejecucion.pickle'

### 5.2 Guardar el pipe final de ejecución

In [45]:
ruta_pipe = ruta_proyecto + '/04_Modelos/' + nombre_pipe

# Guardado manual
with open(ruta_pipe, mode='wb') as file:
   cloudpickle.dump(pipe_ejecucion, file)

El pipeline de ejecución lo usaremos pasándole:
* Datos iniciales + los cambios de estructura de Pandas
* Aplicar pipeline de ejecución